# 03 - Avaliação dos Modelos de Segmentação

Calcula métricas de desempenho (IoU, similaridade de área e perímetro) e gera ranking dos modelos de segmentação.

---

## 1. Setup

Configura ambiente, importa bibliotecas e aplica estilo de visualização.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from src.analysis import MetricsCollector, ModelRanker
from src.visualization import *
from src.config import RANKING_WEIGHTS

import matplotlib.pyplot as plt
%matplotlib inline

setup_plot_style()
print("✓ Setup completo")

---

## 2. Coletar Métricas

Processa todas as imagens e calcula métricas de similaridade entre segmentações e ground truth.

In [ ]:
collector = MetricsCollector(force_recalculate=False)
df_metrics = collector.collect_all_metrics()

print(f"\nResumo:")
print(f"  Imagens: {df_metrics['nome_arquivo'].nunique()}")
print(f"  Modelos: {df_metrics['modelo'].nunique()}")
print(f"  Total de avaliações: {len(df_metrics)}")

### 2.1 Estatísticas Resumidas

Calcula médias das métricas agrupadas por modelo.

In [ ]:
# Médias por modelo
stats = df_metrics.groupby('modelo')[['iou', 'area_similarity', 'perimetro_similarity']].mean()
stats = stats.sort_values('iou', ascending=False)
stats['area_sim_pct'] = stats['area_similarity'] * 100
stats['perim_sim_pct'] = stats['perimetro_similarity'] * 100

print("\nMétricas Médias por Modelo:")
print(stats[['iou', 'area_sim_pct', 'perim_sim_pct']].to_string())

---

## 3. Análise de IoU

IoU mede a sobreposição entre a segmentação e o ground truth.
**Maior é melhor** (0 a 1).

In [ ]:
fig = plot_iou_analysis(df_metrics)
plt.show()

### Top 5 - Maior IoU

Exibe as 5 segmentações com maior Intersection over Union.

In [ ]:
top_iou, bottom_iou = get_top_bottom_iou(df_metrics, n=5)
print("\nTop 5 Melhores Resultados:")
print(top_iou[['nome_arquivo', 'modelo', 'iou']].to_string(index=False))

fig = plot_image_grid(top_iou, df_metrics, "Top 5 - Maior IoU", "iou", max_images=5)
plt.show()

### Bottom 5 - Menor IoU

Exibe as 5 segmentações com menor Intersection over Union.

In [ ]:
fig = plot_image_grid(bottom_iou, df_metrics, "Bottom 5 - Menor IoU", "iou", max_images=5)
plt.show()

---

## 4. Análise de Área

Similaridade entre área prevista e ground truth.
**Maior é melhor** (100% = perfeito).

In [ ]:
fig = plot_area_analysis(df_metrics)
plt.show()

### Top 5 - Maior Similaridade

Exibe as 5 segmentações com maior similaridade de área.

In [ ]:
top_area, bottom_area = get_top_bottom_area(df_metrics, n=5)
print("\nTop 5 - Maior Similaridade:")
print(top_area[['nome_arquivo', 'modelo', 'area_similarity']].to_string(index=False))

fig = plot_image_grid(top_area, df_metrics, "Top 5 - Maior Similaridade de Área", "area_similarity", max_images=5)
plt.show()

### Bottom 5 - Menor Similaridade

Exibe as 5 segmentações com menor similaridade de área.

In [ ]:
fig = plot_image_grid(bottom_area, df_metrics, "Bottom 5 - Menor Similaridade de Área", "area_similarity", max_images=5)
plt.show()

---

## 5. Análise de Perímetro

Similaridade entre perímetro previsto e ground truth.
**Maior é melhor** (100% = perfeito).

In [ ]:
fig = plot_perimetro_analysis(df_metrics)
plt.show()

### Top 5 - Maior Similaridade

Exibe as 5 segmentações com maior similaridade de perímetro.

In [ ]:
top_perim, bottom_perim = get_top_bottom_perimetro(df_metrics, n=5)
print("\nTop 5 - Maior Similaridade:")
print(top_perim[['nome_arquivo', 'modelo', 'perimetro_similarity']].to_string(index=False))

fig = plot_image_grid(top_perim, df_metrics, "Top 5 - Maior Similaridade de Perímetro", "perimetro_similarity", max_images=5)
plt.show()

### Bottom 5 - Menor Similaridade

Exibe as 5 segmentações com menor similaridade de perímetro.

In [ ]:
fig = plot_image_grid(bottom_perim, df_metrics, "Bottom 5 - Menor Similaridade de Perímetro", "perimetro_similarity", max_images=5)
plt.show()

---

## 6. Ranking Final

Calcula score ponderado combinando IoU, similaridade de área e perímetro, e gera ranking dos modelos.

In [ ]:
print("\nPesos configurados:")
for metric, weight in RANKING_WEIGHTS.items():
    print(f"  {metric}: {weight:.0%}")

In [ ]:
ranker = ModelRanker(df_metrics, weights=RANKING_WEIGHTS)
df_ranking = ranker.calculate_ranking()

print("\n" + "="*70)
print("RANKING FINAL")
print("="*70)
print(df_ranking[['rank', 'modelo', 'score']].to_string(index=False))

In [ ]:
fig = plot_ranking_analysis(df_ranking)
plt.show()

### Melhor Modelo

Exibe métricas detalhadas do modelo com melhor desempenho.

In [ ]:
best = df_ranking.iloc[0]
print("\n MELHOR MODELO ")
print("="*70)
print(f"Modelo: {best['modelo']}")
print(f"Score: {best['score']:.4f}")
print(f"\nMétricas:")
print(f"  IoU médio: {best['iou_mean']:.4f}")
print(f"  Similaridade de área: {best['area_similarity_mean']*100:.2f}%")
print(f"  Similaridade de perímetro: {best['perimetro_similarity_mean']*100:.2f}%")
print("="*70)